## TASK 4: Distributed Computing

In [61]:
!pip install pyngrok

In [1]:
import time
from pyngrok import ngrok
from pyspark.sql import SparkSession
from pyspark.storagelevel import StorageLevel

In [2]:
NGROK_TOKEN = "3FlCcB7M0eiNFHmcG8leFa6vNdK_85cxc24pUjmvUQZp3XitQ"
ngrok.set_auth_token(NGROK_TOKEN)

In [3]:
spark_session = (
    SparkSession.builder
    .appName("Task_4_Distributed_Computing_Session")
    .config("spark.driver.memory", "4g")            # Explicit 4G allocation
    .config("spark.executor.memory", "4g")          # Explicit 4G worker allocation
    .config("spark.sql.shuffle.partitions", "200")  # Scaled partitions to solve structural data skew
    .config("spark.default.parallelism", "200")     # Synchronized default execution slots
    .config("spark.ui.port", "4040")                # Force explicit port binding
    .getOrCreate()
)

In [4]:
try:
    active_tunnels = ngrok.get_tunnels()
    for tunnel in active_tunnels:
        ngrok.disconnect(tunnel.public_url)

    public_url = ngrok.connect(4040)
    print("\n" + "="*80)
    print(f"CLICK THIS URL FOR YOUR TASK 4 REPORT SCREENSHOTS: {public_url.public_url}")
    print("="*80 + "\n")
except Exception as e:
    print(f"Tunnel environment bridge failure: {e}")


CLICK THIS URL FOR YOUR TASK 4 REPORT SCREENSHOTS: https://shiftless-pacify-chug.ngrok-free.dev



In [5]:
print("Reading preprocessed transaction datasets into storage dataframes...")

training_dataframe = spark_session.read.parquet(
    "data/processed/training_dataset"
)

testing_dataframe = spark_session.read.parquet(
    "data/processed/testing_dataset"
)

Reading preprocessed transaction datasets into storage dataframes...


In [6]:
train_count = training_dataframe.count()
test_count = testing_dataframe.count()
print(f"Data Loaded. training_dataframe Row Count: {train_count:,}")
print(f"Data Loaded. testing_dataframe Row Count: {test_count:,}")

Data Loaded. training_dataframe Row Count: 16,799,975
Data Loaded. testing_dataframe Row Count: 4,200,025


In [7]:
print("EXECUTING INDEPENDENT DISTRIBUTED PERFORMANCE STRATEGIES")

EXECUTING INDEPENDENT DISTRIBUTED PERFORMANCE STRATEGIES


In [8]:
print("Step 4.3: Apportioning dataset layout to 300 partitions grouped via 'isFraud' column key...")
balanced_training_df = training_dataframe.repartition(300, "isFraud")

Step 4.3: Apportioning dataset layout to 300 partitions grouped via 'isFraud' column key...


In [9]:
# Optimization 2: Explicit structural layout persistence tier selection
print("Step 4.2: Persisting operational arrays via DISK_ONLY storage configurations...")
balanced_training_df.persist(StorageLevel.DISK_ONLY)

Step 4.2: Persisting operational arrays via DISK_ONLY storage configurations...


DataFrame[type: string, amount: double, oldbalanceOrg: double, newbalanceOrig: double, oldbalanceDest: double, newbalanceDest: double, isFraud: bigint, isFlaggedFraud: bigint, transactionTypeIndex: double, assembledFeatures: vector, features: vector]

In [10]:
print("\nExecuting baseline action invocation to populate storage cache mapping layer...")
start_time = time.time()
first_count = balanced_training_df.count()
print(f"First Execution Duration (Cache Miss): {time.time() - start_time:.2f} seconds")


Executing baseline action invocation to populate storage cache mapping layer...
First Execution Duration (Cache Miss): 156.89 seconds


In [11]:
print("\nExecuting subsequent action invocation reading directly from disk blocks...")
start_time = time.time()
second_count = balanced_training_df.count()
print(f"Second Execution Duration (Cache Hit): {time.time() - start_time:.2f} seconds")


Executing subsequent action invocation reading directly from disk blocks...
Second Execution Duration (Cache Hit): 9.33 seconds
